In [1]:
# This notebook is a replication of Gu, Kelly, and Xiu (2020) Emprical Asset Pricing using Machine Learning;
# 30-year stock excess return predictions (1987-2016) were generated from 3-hidden-layer neural networks in notebook 2_NN3;
# Here, stocks are decile-sorted based on their predicted returns each month, using equally-weighted and value-weighted methods;
# Then, a dollar-neutral (zero-net investment) strategy is adopted: long highest decile, short lowest decile;
# All results can be found at the end of the notebook.

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from numpy.linalg import inv


## Stock permnos, market values saved from 1_Preprocessing
permno = np.load('permno.npy')
mdate = np.load('oos_periods.npy', allow_pickle=True)
mktval = np.load('mvel.npy')
## Stock excess returns predictions, saved from 2_NN3
y_true_rep = torch.load('y_true_rep.pth').cpu()
y_pred_rep = torch.load('y_pred_rep_NN3.pth').cpu()

## Aggregate stock-level monthly prediction
stock_level_prediction = pd.DataFrame({
    'mdate': mdate,
    'permno': permno,
    'mktval': mktval,
})
stock_level_prediction = stock_level_prediction.drop(index=stock_level_prediction[stock_level_prediction['mdate']>'2016-12'].index) #extract replication period
stock_level_prediction['y_pred'] = y_pred_rep.ravel()
stock_level_prediction['y_true'] = y_true_rep.ravel()
T = stock_level_prediction['mdate'].nunique()
stock_level_prediction['decile'] = stock_level_prediction.groupby('mdate')['y_pred'].transform(lambda x: pd.qcut(x, 10, labels=False))
stock_level_prediction['sum_mval'] = stock_level_prediction.groupby(['mdate', 'decile'])['mktval'].transform('sum')

topdecile_idx = stock_level_prediction[stock_level_prediction['decile'] == 9].index
botdecile_idx = stock_level_prediction[stock_level_prediction['decile'] == 0].index
mid_idx = stock_level_prediction[~stock_level_prediction['decile'].isin([0, 9])].index
print(f'Stock-level monthly predictions:')
print(f'Shape: {stock_level_prediction.shape}')
print(stock_level_prediction.head())
print(stock_level_prediction.tail())
print(f'--------------------------------------------------------------------------------------------------------------------------------------------------------')

# Portfolio return------------------------------------------------------------------------------------------------------------------------------------------------
## Equally weighted decile-sorted portfolios
ewportf_monthly = pd.DataFrame({
    'y_pred': stock_level_prediction.groupby(['mdate', 'decile'])['y_pred'].mean(),
    'y_true': stock_level_prediction.groupby(['mdate', 'decile'])['y_true'].mean(),
})
ewportf_monthly = ewportf_monthly.reset_index(drop=False)
ewportf_monthly_ret = ewportf_monthly.pivot(index='mdate', columns='decile', values='y_true').reset_index(
    drop=False).rename_axis(None, axis=1)
ewportf_monthly_pred = ewportf_monthly.pivot(index='mdate', columns='decile', values='y_pred').reset_index(
    drop=False).rename_axis(None, axis=1)
ewportf_monthly_pred['H-L'] = ewportf_monthly_pred[9] - ewportf_monthly_pred[0]
ewportf_monthly_ret['H-L'] = ewportf_monthly_ret[9] - ewportf_monthly_ret[0]

## Value weighted decile-sorted portfolios
vwportf_monthly = pd.DataFrame({
    'y_pred': (stock_level_prediction['y_pred'] * stock_level_prediction['mktval'] / stock_level_prediction[
        'sum_mval']).groupby([stock_level_prediction['mdate'], stock_level_prediction['decile']]).sum(),
    'y_true': (stock_level_prediction['y_true'] * stock_level_prediction['mktval'] / stock_level_prediction[
        'sum_mval']).groupby([stock_level_prediction['mdate'], stock_level_prediction['decile']]).sum()
})
vwportf_monthly = vwportf_monthly.reset_index(drop=False)
vwportf_monthly_ret = vwportf_monthly.pivot(index='mdate', columns='decile', values='y_true').reset_index(
    drop=False).rename_axis(None, axis=1)
vwportf_monthly_pred = vwportf_monthly.pivot(index='mdate', columns='decile', values='y_pred').reset_index(
    drop=False).rename_axis(None, axis=1)
vwportf_monthly_pred['H-L'] = vwportf_monthly_pred[9] - vwportf_monthly_pred[0]
vwportf_monthly_ret['H-L'] = vwportf_monthly_ret[9] - vwportf_monthly_ret[0]

#Drawdown---------------------------------------------------------------------------------------------------------------------------------------------------
ew_logret = np.log(ewportf_monthly_ret['H-L'] + 1)
ew_peak = np.maximum.accumulate(ew_logret.cumsum())
ew_dd = ew_peak - ew_logret.cumsum()
vw_logret = np.log(vwportf_monthly_ret['H-L'] + 1)
vw_peak = np.maximum.accumulate(vw_logret.cumsum())
vw_dd = vw_peak - vw_logret.cumsum()

# Turnover: Turnover weights are the stock's weights w.r.t the total long+short portfolio value-------------------------------------------------------------
# i.e when stock i in long portfolio has weight w_i, and long/short ratio = 1, stock i's weight w.r.t total portfolio value is w_i/2
ew_weight = 1 / stock_level_prediction['permno'].groupby([stock_level_prediction['mdate'], stock_level_prediction['decile']]).transform('count')
ew_weight[mid_idx] = 0.0
ew_weight[topdecile_idx] = ew_weight[topdecile_idx] * 1/2
ew_weight[botdecile_idx] = -1 * ew_weight[botdecile_idx] * 1/2
lagged_ew_weight = ew_weight.groupby(stock_level_prediction['permno']).shift(1)
ew_numer_lag = lagged_ew_weight * (1 + stock_level_prediction['y_true'])
ew_denom_lag = 1 + (lagged_ew_weight * stock_level_prediction['y_true']).groupby(stock_level_prediction['mdate']).transform('sum')
ew_turnover_t = (ew_weight - ew_numer_lag / ew_denom_lag).abs().groupby(stock_level_prediction['mdate']).sum()

vw_weight = stock_level_prediction['mktval'] / stock_level_prediction['sum_mval']
vw_weight[mid_idx] = 0.0
vw_weight[topdecile_idx] = vw_weight[topdecile_idx] * 1/2
vw_weight[botdecile_idx] = -1 * vw_weight[botdecile_idx] * 1/2
lagged_vw_weight = vw_weight.groupby(stock_level_prediction['permno']).shift(1)
vw_numer_lag = lagged_vw_weight * (1 + stock_level_prediction['y_true'])
vw_denom_lag = 1 + (lagged_vw_weight * stock_level_prediction['y_true']).groupby(stock_level_prediction['mdate']).transform('sum')
vw_turnover_t = (vw_weight - vw_numer_lag / vw_denom_lag).abs().groupby(stock_level_prediction['mdate']).sum()

# Risk-adjusted performance-------------------------------------------------------------------------------------------------------------------------------------
## Fama-French 5 factors + momentum
ff5 = pd.read_csv(
    'F-F_Research_Data_5_Factors_2x3.csv')  #downloaded from https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html
ffmom = pd.read_csv(
    'F-F_Momentum_Factor.csv')  #downloaded from https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html
ff5['mdate'] = pd.to_datetime(ff5.iloc[:, 0].astype(str), format='%Y%m', errors='coerce').dt.to_period('M')
ffmom['mdate'] = pd.to_datetime(ffmom.iloc[:, 0].astype(str), format='%Y%m', errors='coerce').dt.to_period('M')
ff5 = ff5.drop(index=ff5[(ff5['mdate'] < mdate[0]) | (ff5['mdate'] > '2016-12')].index)
ffmom = ffmom.drop(index=ffmom[(ffmom['mdate'] < mdate[0]) | (ffmom['mdate'] > '2016-12')].index)
ff6 = pd.merge(ff5[ff5.columns[1:]], ffmom[ffmom.columns[1:]], how='left', on='mdate')
factors = [col for col in ff6.columns if col not in ['mdate','RF']]
ff6[factors] = ff6[factors] /100
print(f'Fama-French 6 factors, monthly data:')
print(f'Shape: {ff6.shape}')
print(ff6.head())
print(f'--------------------------------------------------------------------------------------------------------------------------------------------------------')

X = np.hstack((np.ones(T).reshape(-1, 1), ff6[factors]))
k = X.shape[1]
ew_y = ewportf_monthly_ret['H-L'].values.reshape(-1, 1)
vw_y = vwportf_monthly_ret['H-L'].values.reshape(-1, 1)

ew_coef = (inv(X.T @ X) @ X.T @ ew_y).reshape(-1, 1)
ew_alpha = ew_coef[0].item()
ew_betas = ew_coef[1:]
ew_alpha_t = ew_y - X[:, 1:] @ ew_betas
ew_alpha_se = np.sqrt(((ew_y - X @ ew_coef) ** 2).sum()/(T-k) * inv(X.T @ X)[0, 0])
ew_alpha_tstat = ew_alpha / ew_alpha_se

vw_coef = (inv(X.T @ X) @ X.T @ vw_y).reshape(-1, 1)
vw_alpha = vw_coef[0].item()
vw_betas = vw_coef[1:]
vw_alpha_t = vw_y - X[:, 1:] @ vw_betas
vw_alpha_se = np.sqrt(((vw_y - X @ vw_coef) ** 2).sum()/(T-k) * inv(X.T @ X)[0, 0])
vw_alpha_tstat = vw_alpha / vw_alpha_se

#Result tables------------------------------------------------------------------------------------------------------------------------------------------------
Table_7_Performance_of_value_weighted_MLportfolios_NN3 = pd.DataFrame(
    data={
    'y_pred': vwportf_monthly_pred.iloc[:,1:].mean(axis=0).values,
    'y_true': vwportf_monthly_ret.iloc[:,1:].mean(axis=0).values,
    'y_std': vwportf_monthly_ret.iloc[:,1:].std(axis=0).values
}, index=[f'decile {i}' for i in range(1,11)] + ['H-L']
)
Table_7_Performance_of_value_weighted_MLportfolios_NN3['sharpe ratio (annualized)'] = Table_7_Performance_of_value_weighted_MLportfolios_NN3['y_true'] * 12 / np.sqrt(Table_7_Performance_of_value_weighted_MLportfolios_NN3['y_std']**2 *12)
Table_7_Performance_of_value_weighted_MLportfolios_NN3[['y_pred','y_true','y_std']] = Table_7_Performance_of_value_weighted_MLportfolios_NN3[['y_pred','y_true','y_std']]*100
Table_7_Comparison_with_Originalpaper = pd.DataFrame(
    Table_7_Performance_of_value_weighted_MLportfolios_NN3.iloc[[0,-2,-1],:].stack(),
    columns=['Replicated']
)
Table_7_Comparison_with_Originalpaper['Original'] = pd.Series([-0.03, -0.43, 7.73, -0.19, 1.83, 1.69, 7.29, 0.80, 1.86, 2.12, 6.13, 1.20],
                                                              index=Table_7_Comparison_with_Originalpaper.index)
Table_7_Comparison_with_Originalpaper['Rep. quality'] = pd.Series(['not good','not good','good','not good','good','good','good','good','not good','not good','good','good'],
                                                                  index=Table_7_Comparison_with_Originalpaper.index)

print(f'Table 7 (in the main paper): Performance of value-weighted machine learning portfolios:')
print(Table_7_Performance_of_value_weighted_MLportfolios_NN3)
print(f'///////////////////////////////////////////')
print(f'Table7: Comparision with original paper:')
print(Table_7_Comparison_with_Originalpaper)
print(f'--------------------------------------------------------------------------------------------------------------------------------------------------------')

Table_A9_Performance_of_equally_weighted_MLportfolios_NN3 = pd.DataFrame({
    'y_pred': ewportf_monthly_pred.iloc[:,1:].mean(axis=0).values,
    'y_true': ewportf_monthly_ret.iloc[:,1:].mean(axis=0).values,
    'y_std': ewportf_monthly_ret.iloc[:,1:].std(axis=0).values,
}, index=[f'decile {i}' for i in range(1,11)] + ['H-L']
)
Table_A9_Performance_of_equally_weighted_MLportfolios_NN3['sharpe ratio (annualized)'] = Table_A9_Performance_of_equally_weighted_MLportfolios_NN3['y_true'] * 12 / np.sqrt(Table_A9_Performance_of_equally_weighted_MLportfolios_NN3['y_std']**2 *12)
Table_A9_Performance_of_equally_weighted_MLportfolios_NN3[['y_pred','y_true','y_std']] = Table_A9_Performance_of_equally_weighted_MLportfolios_NN3[['y_pred','y_true','y_std']]*100
Table_A9_Comparison_with_Originalpaper = pd.DataFrame(
    Table_A9_Performance_of_equally_weighted_MLportfolios_NN3.iloc[[0,-2,-1],:].stack(),
    columns=['Replicated']
)
Table_A9_Comparison_with_Originalpaper['Original'] = pd.Series([-0.31, -0.92, 7.94, -0.40, 2.28, 2.35, 8.11, 1.00, 2.58, 3.27, 4.80, 2.36],
                                                index=Table_A9_Comparison_with_Originalpaper.index)
Table_A9_Comparison_with_Originalpaper['Rep. quality'] = pd.Series(['not good', 'not good', 'good', 'good', 'good', 'good', 'good', 'good', 'good' ,'good', 'good', 'good'],
                                                index=Table_A9_Comparison_with_Originalpaper.index)

print(f'Table A9 (in the paper\'s appendix): Performance of equally-weighted machine learning portfolios:')
print(Table_A9_Performance_of_equally_weighted_MLportfolios_NN3)
print(f'///////////////////////////////////////////')
print(f'Table A9: Comparision with original paper:')
print(Table_A9_Comparison_with_Originalpaper)
print(f'--------------------------------------------------------------------------------------------------------------------------------------------------------')

Table_8_Drawdown_Turnover_Risk_adjusted_return_of_MLportfolios_NN3 = pd.DataFrame(
    data=np.vstack((
        [vw_dd.max()*100, 30.84], [vw_logret.min()*-100, 30.84], [vw_turnover_t.mean()*100, 123.50], [vwportf_monthly_ret['H-L'].mean()*100, 2.12], [vw_alpha*100, 1.52], [vw_alpha_tstat, 4.92],
        [ew_dd.max()*100, 17.34], [ew_logret.min()*-100, 12.50], [ew_turnover_t.mean()*100, 113.76], [ewportf_monthly_ret['H-L'].mean()*100, 3.27], [ew_alpha*100, 3.02], [ew_alpha_tstat, 11.70]
    )),
    columns=['Replicated', 'Original'],
    index=pd.MultiIndex.from_product([['Value weighted', 'Equally weighted'],('Max drawdown (%)', 'Max 1M loss (%)', 'Turnover (%)', 'Mean return (%)', 'Risk-adjusted return (alpha) (%)','t_stat alpha')])
)
Table_8_Drawdown_Turnover_Risk_adjusted_return_of_MLportfolios_NN3['Rep. quality'] = pd.Series(['good','good','good','not good','not good','good','good','good','good','good','good','good'], index=Table_8_Drawdown_Turnover_Risk_adjusted_return_of_MLportfolios_NN3.index)
print(f'Table 8 (in the main paper): Drawdown, turnover, risk-adjusted return (alpha):')
print(Table_8_Drawdown_Turnover_Risk_adjusted_return_of_MLportfolios_NN3)

Stock-level monthly predictions:
Shape: (2494470, 7)
     mdate  permno        mktval    y_pred    y_true  decile      sum_mval
0  1987-01   10000   1981.546875  0.002911 -0.216321       2  7.400717e+07
1  1987-01   10001   6937.000000  0.004099 -0.039914       4  1.071027e+08
2  1987-01   10002  14540.625000  0.004170  0.091760       5  2.258930e+08
3  1987-01   10003  42061.250000  0.003164  0.125670       3  6.734152e+07
4  1987-01   10005    433.687500  0.002033  0.662467       1  8.040712e+07
           mdate  permno        mktval    y_pred    y_true  decile  \
2494465  2016-12   93427  1.577480e+06  0.010508 -0.058711       8   
2494466  2016-12   93428  1.250976e+06  0.004295 -0.006324       3   
2494467  2016-12   93429  5.600537e+06  0.006576  0.072124       5   
2494468  2016-12   93434  8.573280e+04  0.011637 -0.041967       9   
2494469  2016-12   93436  2.840318e+07 -0.002398  0.127947       0   

             sum_mval  
2494465  2.657203e+09  
2494466  3.925804e+09  
2494